# Atividade - Feature Engineering e PCA
Este notebook segue o roteiro da atividade prática: seleção de atributos, limpeza, pivotamento, normalização, PCA e respostas das 5 questões.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import unicodedata

## 1. Carregamento dos dados e definição do período

In [ ]:
def remove_accents(input_str):
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    return ''.join([c for c in nfkd_form if not unicodedata.combining(c)])

base_dir = Path('../7- Feature Engineering amp Redução de Dimensionalidade-20260503')
arq_original = base_dir / 'WDICSV.csv'
dicionario = base_dir / 'WDISeries.csv'
paises = base_dir / 'WDICountry.csv'

anos_g1 = ['2014', '2015', '2016', '2017', '2018']
anos_g2 = ['2019', '2020', '2021', '2022', '2023']

# Ajuste para o seu grupo: use anos_g1 ou anos_g2
anos_analise = anos_g2

data = pd.read_csv(arq_original)
series_dict = pd.read_csv(dicionario)
paises_dict = pd.read_csv(paises)
paises_dict['Table Name'] = paises_dict['Table Name'].apply(remove_accents)
paises_dict = paises_dict.set_index(['Country Code', 'Table Name'])

print('Formato dos dados brutos:', data.shape)
print('Total de indicadores no dicionário:', series_dict.shape[0])

## 2. Seleção de features e pivotamento

In [ ]:
# Features escolhidas por pilar (12 no total)
features_map = {
    'GDP': 'GDP (current US$)',
    'Inflation': 'Inflation, consumer prices (annual %)',
    'Exports': 'Exports of goods and services (% of GDP)',
    'Life_expectancy': 'Life expectancy at birth, total (years)',
    'Health_expenditure_per_capita': 'Current health expenditure per capita (current US$)',
    'Internet_users': 'Individuals using the Internet (% of population)',
    'Literacy_rate': 'Literacy rate, adult total (% of people ages 15 and above)',
    'Urban_population': 'Urban population (% of total population)',
    'CO2_emissions': 'CO2 emissions (metric tons per capita)',
    'Energy_consumption': 'Electric power consumption (kWh per capita)',
    'Total_population': 'Population, total',
    'Poverty_rate': 'Poverty headcount ratio at $8.30 a day (2021 PPP) (% of population)'
}

features_disponiveis = [v for v in features_map.values() if v in data['Indicator Name'].unique()]
features_ausentes = [v for v in features_map.values() if v not in features_disponiveis]

print(f'Features disponíveis: {len(features_disponiveis)}')
if features_ausentes:
    print('Features não encontradas no arquivo:')
    for f in features_ausentes:
        print('-', f)

data_reduzida = data[data['Indicator Name'].isin(features_disponiveis)].copy()
data_reduzida['Mean'] = data_reduzida[anos_analise].mean(axis=1)

df = data_reduzida.pivot(
    index=['Country Code', 'Country Name'],
    columns='Indicator Name',
    values='Mean'
).copy()

# Renomeia para nomes curtos
rename_inverse = {v: k for k, v in features_map.items()}
df = df.rename(columns=rename_inverse)

print('Formato após pivotamento:', df.shape)
display(df.head())

## 3. Tratamento de dados faltantes

In [ ]:
ausencia_percentual = (df.isna().mean() * 100).sort_values(ascending=False)
display(ausencia_percentual)

# Mantém linhas com pelo menos 60% das colunas preenchidas
df = df.dropna(thresh=int(df.shape[1] * 0.6))

# Imputação por mediana em colunas numéricas
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(df[col].median())

print('Formato após limpeza:', df.shape)
print('Total de nulos restantes:', int(df.isna().sum().sum()))

## 4. Normalização e aplicação de PCA 

In [ ]:
def pca_svd(X: pd.DataFrame):
    Xc = X - X.mean(axis=0)
    U, S, VT = np.linalg.svd(Xc.values, full_matrices=False)
    componentes = VT.T
    scores = Xc.values @ componentes
    var = (S ** 2) / (len(Xc) - 1)
    var_explicada = var / var.sum()
    return componentes, scores, var_explicada

# Remove possíveis infinitos
df = df.replace([np.inf, -np.inf], np.nan).dropna()

# Z-score
df_norm = (df - df.mean()) / df.std(ddof=0)
df_norm = df_norm.replace([np.inf, -np.inf], np.nan).dropna()

componentes_base, scores_base, var_explicada_base = pca_svd(df_norm)

pca_df_base = pd.DataFrame(
    scores_base[:, :2],
    index=df_norm.index,
    columns=['PC1', 'PC2']
  )

print(f"Variância explicada - PC1: {var_explicada_base[0]:.2%}")
print(f"Variância explicada - PC2: {var_explicada_base[1]:.2%}")
print(f"Variância explicada acumulada (2 PCs): {(var_explicada_base[0] + var_explicada_base[1]):.2%}")

plt.figure(figsize=(10, 7))
plt.scatter(pca_df_base['PC1'], pca_df_base['PC2'], alpha=0.6)
plt.axhline(0, color='grey', lw=1, ls='--')
plt.axvline(0, color='grey', lw=1, ls='--')
plt.xlabel(f"PC1 ({var_explicada_base[0]:.2%})")
plt.ylabel(f"PC2 ({var_explicada_base[1]:.2%})")
plt.title('PCA base - Dispersão por país')
plt.show()

## 5. Questões da atividade

### Questão 1 - Interpretação do PCA
A seguir, avaliamos os pesos (loadings) de PC1 e PC2 para interpretar o significado dos componentes.

In [ ]:
loadings = pd.DataFrame(
    componentes_base[:, :2],
    index=df_norm.columns,
    columns=['PC1', 'PC2']
 )

top_pc1 = loadings['PC1'].abs().sort_values(ascending=False).head(5)
top_pc2 = loadings['PC2'].abs().sort_values(ascending=False).head(5)

print('Top 5 contribuições para PC1:')
display(loadings.loc[top_pc1.index].sort_values(by='PC1', key=np.abs, ascending=False))

print('Top 5 contribuições para PC2:')
display(loadings.loc[top_pc2.index].sort_values(by='PC2', key=np.abs, ascending=False))

sintese_pc1 = ', '.join(top_pc1.index.tolist()[:3])
sintese_pc2 = ', '.join(top_pc2.index.tolist()[:3])
print(f'Síntese sugerida PC1: eixo associado a {sintese_pc1}.')
print(f'Síntese sugerida PC2: eixo associado a {sintese_pc2}.')

### Questão 2 - Índice de Eficiência de Saúde
Criamos a feature Eficiencia_Saude = Life_expectancy / Health_expenditure_per_capita.

In [ ]:
if {'Life_expectancy', 'Health_expenditure_per_capita'}.issubset(df.columns):
    df['Eficiencia_Saude'] = df['Life_expectancy'] / df['Health_expenditure_per_capita'].replace(0, np.nan)
    df['Eficiencia_Saude'] = df['Eficiencia_Saude'].replace([np.inf, -np.inf], np.nan).fillna(df['Eficiencia_Saude'].median())
    print('Maiores eficiências de saúde:')
    display(df['Eficiencia_Saude'].sort_values(ascending=False).head(10))
else:
    print('Colunas necessárias para Eficiencia_Saude não estão disponíveis.')

### Questão 3 - Binarização e categorização (discretização)
Criamos a variável Nivel_Industrial usando a soma de Exportacoes e Consumo de Energia.

In [ ]:
if {'Exports', 'Energy_consumption'}.issubset(df.columns):
    score_industrial = df['Exports'] + df['Energy_consumption']
    p25 = score_industrial.quantile(0.25)
    p75 = score_industrial.quantile(0.75)

    df['Nivel_Industrial'] = np.where(
        score_industrial >= p75, 'Alta',
        np.where(score_industrial <= p25, 'Baixa', 'Media')
    )

    mapa_cores = {'Baixa': '#1f77b4', 'Media': '#ff7f0e', 'Alta': '#2ca02c'}
    cores = df.loc[pca_df_base.index, 'Nivel_Industrial'].map(mapa_cores)

    plt.figure(figsize=(10, 7))
    plt.scatter(pca_df_base['PC1'], pca_df_base['PC2'], c=cores, alpha=0.65)
    plt.axhline(0, color='grey', lw=1, ls='--')
    plt.axvline(0, color='grey', lw=1, ls='--')
    plt.xlabel(f"PC1 ({var_explicada_base[0]:.2%})")
    plt.ylabel(f"PC2 ({var_explicada_base[1]:.2%})")
    plt.title('PCA colorido por Nivel_Industrial')
    plt.show()

    print(df['Nivel_Industrial'].value_counts())
else:
    print('Colunas necessárias para Nivel_Industrial não estão disponíveis.')

### Questão 4 - Normalização condicional per capita
Normalizamos métricas absolutas dividindo por Total_population e comparamos o PCA antes e depois.

In [ ]:
df_pc = df.copy()

if 'Total_population' in df_pc.columns:
    colunas_consumo_total = [c for c in ['GDP'] if c in df_pc.columns]

    for c in colunas_consumo_total:
        df_pc[c] = df_pc[c] / df_pc['Total_population'].replace(0, np.nan)

    df_pc = df_pc.replace([np.inf, -np.inf], np.nan).dropna()
    df_pc_norm = (df_pc.select_dtypes(include=[np.number]) - df_pc.select_dtypes(include=[np.number]).mean()) / df_pc.select_dtypes(include=[np.number]).std(ddof=0)
    df_pc_norm = df_pc_norm.replace([np.inf, -np.inf], np.nan).dropna()

    componentes_pc, scores_pc, var_explicada_pc = pca_svd(df_pc_norm)
    pca_df_pc = pd.DataFrame(scores_pc[:, :2], index=df_pc_norm.index, columns=['PC1', 'PC2'])

    print(f"PC1 base: {var_explicada_base[0]:.2%}")
    print(f"PC1 per capita: {var_explicada_pc[0]:.2%}")
    print('Mudança em PC1:', f"{(var_explicada_pc[0]-var_explicada_base[0]):.2%}")

    plt.figure(figsize=(10, 7))
    plt.scatter(pca_df_pc['PC1'], pca_df_pc['PC2'], alpha=0.6, color='#9467bd')
    plt.axhline(0, color='grey', lw=1, ls='--')
    plt.axvline(0, color='grey', lw=1, ls='--')
    plt.xlabel(f"PC1 ({var_explicada_pc[0]:.2%})")
    plt.ylabel(f"PC2 ({var_explicada_pc[1]:.2%})")
    plt.title('PCA após normalização per capita')
    plt.show()
else:
    print('Coluna Total_population não está disponível.')

### Questão 5 - Tratamento de skewness (assimetria)
Comparamos o histograma de PIB antes e depois da transformação logarítmica.

In [ ]:
if 'GDP' in df.columns:
    gdp_raw = df['GDP'].dropna().copy()
    gdp_log = np.log1p(gdp_raw.clip(lower=0))

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.hist(gdp_raw, bins=30, color='#1f77b4', alpha=0.8)
    plt.title('PIB - distribuição original')
    plt.xlabel('GDP')
    plt.ylabel('Frequência')

    plt.subplot(1, 2, 2)
    plt.hist(gdp_log, bins=30, color='#ff7f0e', alpha=0.8)
    plt.title('PIB - distribuição log1p')
    plt.xlabel('log1p(GDP)')
    plt.ylabel('Frequência')

    plt.tight_layout()
    plt.show()

    print(f'Assimetria (skewness) PIB original: {gdp_raw.skew():.2f}')
    print(f'Assimetria (skewness) PIB log: {gdp_log.skew():.2f}')
else:
    print('Coluna GDP não está disponível para análise de skewness.')